In [3]:
import pandas as pd
import numpy as np
import os

# ========== 1. 读取所有IoT设备文件 ==========
data_dir = "./"  

files = {
    "weather": "Train_Test_IoT_Weather.csv",
    "modbus": "Train_Test_IoT_Modbus.csv",
    "thermostat": "Train_Test_IoT_Thermostat.csv",
    "fridge": "Train_Test_IoT_Fridge.csv",
    "garage_door": "Train_Test_IoT_Garage_Door.csv",
    "gps": "Train_Test_IoT_GPS_Tracker.csv",
    "motion_light": "Train_Test_IoT_Motion_Light.csv",
}

dfs = {}
for name, filename in files.items():
    df = pd.read_csv(os.path.join(data_dir, filename))
    # 清理列名空格
    df.columns = df.columns.str.strip()
    if 'time' in df.columns:
        df['time'] = df['time'].astype(str).str.strip()
    if 'date' in df.columns:
        df['date'] = df['date'].astype(str).str.strip()
    
    # 构造时间戳
    df['timestamp'] = pd.to_datetime(df['date'] + ' ' + df['time'], format='%d-%b-%y %H:%M:%S')
    
    feature_cols = [c for c in df.columns if c not in ['date', 'time', 'timestamp', 'label', 'type']]
    # 关键修复：特征强制转数字，非法值置NaN
    for col in feature_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # 特征加前缀区分设备
    df = df.rename(columns={c: f"{name}_{c}" for c in feature_cols})
    
    dfs[name] = df
    print(f"[{name}] 样本数: {len(df)}, 时间范围: {df['timestamp'].min()} ~ {df['timestamp'].max()}")

# ========== 2. 按秒级时间戳聚合 ==========
# Weather聚合，均值开启numeric_only
weather_agg = dfs['weather'].groupby('timestamp').agg({
    'weather_temperature': lambda x: x.mean(numeric_only=True),
    'weather_pressure': lambda x: x.mean(numeric_only=True),
    'weather_humidity': lambda x: x.mean(numeric_only=True),
    'label': 'max',
    'type': lambda x: x.mode()[0] if len(x.mode()) > 0 else 'normal'
}).reset_index()

agg_dfs = {'weather': weather_agg}
# 其余设备批量聚合
for name in ['modbus', 'thermostat', 'fridge', 'garage_door', 'gps', 'motion_light']:
    feature_cols = [c for c in dfs[name].columns if c.startswith(f"{name}_")]
    agg = dfs[name].groupby('timestamp').agg(
        {
            **{c: lambda x: x.mean(numeric_only=True) for c in feature_cols},
            'label': 'max',
            'type': lambda x: x.mode()[0] if len(x.mode()) > 0 else 'normal'
        }
    ).reset_index()
    agg = agg.rename(columns={'label': f'{name}_label', 'type': f'{name}_type'})
    agg_dfs[name] = agg

# ========== 3. 时间戳外连接合并 ==========
merged = agg_dfs['weather']
for name in ['modbus', 'thermostat', 'fridge', 'garage_door', 'gps', 'motion_light']:
    merged = pd.merge(merged, agg_dfs[name], on='timestamp', how='outer', suffixes=('', f'_{name}'))

# ========== 4. 统一攻击标签 ==========
label_cols = [c for c in merged.columns if c.endswith('_label')] + ['label']
merged['final_label'] = merged[label_cols].max(axis=1)

type_cols = [c for c in merged.columns if c.endswith('_type')] + ['type']
def get_attack_type(row):
    for c in type_cols:
        val = row[c]
        if pd.notna(val) and val != 'normal':
            return val
    return 'normal'
merged['final_type'] = merged.apply(get_attack_type, axis=1)

# ========== 5. 缺失值填充 ==========
merged = merged.sort_values('timestamp').reset_index(drop=True)
feature_cols = [c for c in merged.columns if c not in 
                ['timestamp', 'final_label', 'final_type', 'label', 'type'] + label_cols + type_cols]
merged[feature_cols] = merged[feature_cols].ffill().fillna(0)

# ========== 6. 输出 ==========
print("\n===== 合并结果 =====")
print(f"总时间步: {len(merged)}")
print(f"特征维度: {len(feature_cols)}")
print(f"攻击样本占比: {merged['final_label'].mean():.2%}")
print(f"攻击类型分布:\n{merged['final_type'].value_counts()}")

merged.to_csv(os.path.join(data_dir, "iot_fusion_weather_all.csv"), index=False)
print("\n已保存: iot_fusion_weather_all.csv")

[weather] 样本数: 39260, 时间范围: 2019-03-31 12:40:22 ~ 2019-04-28 13:59:39
[modbus] 样本数: 31106, 时间范围: 2019-03-31 12:40:58 ~ 2019-04-28 12:40:18
[thermostat] 样本数: 32774, 时间范围: 2019-03-31 12:43:33 ~ 2019-04-28 22:12:49
[fridge] 样本数: 39944, 时间范围: 2019-03-31 12:36:52 ~ 2019-04-29 01:13:08
[garage_door] 样本数: 39587, 时间范围: 2019-03-31 12:41:32 ~ 2019-04-28 23:55:18
[gps] 样本数: 38960, 时间范围: 2019-03-31 12:36:52 ~ 2019-04-28 11:44:04
[motion_light] 样本数: 39488, 时间范围: 2019-03-31 12:36:52 ~ 2019-04-28 21:26:44

===== 合并结果 =====
总时间步: 73902
特征维度: 17
攻击样本占比: 85.90%
攻击类型分布:
final_type
backdoor      25042
password      13217
ddos          11520
normal        10417
injection      6524
ransomware     4170
scanning       1723
xss            1289
Name: count, dtype: int64

已保存: iot_fusion_weather_all.csv


## UltraLite 轻量模型（Mean Teacher 自蒸馏，**不覆盖** `best_innovative_model.pth`）



In [ ]:
# ===== GAT-Mamba UltraLite（second；可单独运行）=====
# 输出: best_gat_mamba_ultralite.pth / best_gat_mamba_ultralite_results.txt
# 依赖: iot_fusion_weather_all.csv
# 结构: d_model=16 / 1×GAT(heads=1) / 单 SimpleMamba / 无 Transformer
# 跨设备语义: 活性加权共识 + 软融合(γ小初始化≈恒等)，不硬压设备私有残差
# 训练: Mean Teacher 自蒸馏（EMA 软标签 + consistency ramp-up）
# 超参(=此前): clf=A, lr=1e-3, dropout=0.1, batch=64, patience=5, ReduceLROnPlateau
# 数据协议(=本目录主实验): window=128 / train_step=1 / eval_step=2

import os
import math
import shutil
import tempfile
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, f1_score, accuracy_score, confusion_matrix
from sklearn.model_selection import StratifiedShuffleSplit
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from torch_geometric.nn import GATv2Conv

_ROOT = r'E:\apt\YES\2'
os.chdir(_ROOT)
assert os.path.isfile(os.path.join(_ROOT, 'iot_fusion_weather_all.csv')), _ROOT
_TMP = os.path.join(_ROOT, '_tmp')
os.makedirs(_TMP, exist_ok=True)
for _k in ('TEMP', 'TMP', 'TMPDIR'):
    os.environ[_k] = _TMP
os.environ['TORCH_HOME'] = os.path.join(_TMP, 'torch')
os.environ['MPLCONFIGDIR'] = os.path.join(_TMP, 'matplotlib')
os.makedirs(os.environ['TORCH_HOME'], exist_ok=True)
os.makedirs(os.environ['MPLCONFIGDIR'], exist_ok=True)
tempfile.tempdir = _TMP

LITE_CKPT = os.path.join(_ROOT, 'best_gat_mamba_ultralite.pth')
LITE_RESULT = os.path.join(_ROOT, 'best_gat_mamba_ultralite_results.txt')
LITE_CM = os.path.join(_ROOT, 'confusion_matrix_ultralite_test.png')
DATA_CSV = os.path.join(_ROOT, 'iot_fusion_weather_all.csv')
LAM_SELF_KD = 0.65      # Mean Teacher 软标签权重；硬损失权重 = 1 - LAM_SELF_KD
LAM_F1 = 0.15
EMA_DECAY = 0.999
RAMP_EPOCHS = 10
KD_TEMP = 2.0           


class IoTFusionDataset(Dataset):
    def __init__(self, df, window_size=128, step=1, scaler=None,
                 fit_scaler=False, label_encoder=None):
        self.window_size = window_size
        self.step = step
        self.label_encoder = label_encoder
        device_prefixes = ['weather', 'modbus', 'thermostat', 'fridge',
                           'garage_door', 'gps', 'motion_light']
        self.devices = device_prefixes
        self.num_devices = len(device_prefixes)
        self.device_feat_cols = {}
        self.device_dims = []
        self.all_feat_cols = []
        for dev in device_prefixes:
            cols = [c for c in df.columns if c.startswith(dev + '_') and
                    not c.endswith('_label') and not c.endswith('_type')]
            self.device_feat_cols[dev] = cols
            self.device_dims.append(len(cols))
            self.all_feat_cols.extend(cols)

        X = df[self.all_feat_cols].values.astype(np.float32)
        y_enc = df['label_encoded'].values.astype(np.int64)
        if scaler is None and fit_scaler:
            self.scaler = StandardScaler()
            X = self.scaler.fit_transform(X)
        elif scaler is not None:
            self.scaler = scaler
            X = self.scaler.transform(X)
        else:
            self.scaler = None
        X = np.nan_to_num(np.clip(X, -8, 8), nan=0.0).astype(np.float32)

        samples, labels = [], []
        for i in range(0, len(X) - window_size + 1, step):
            samples.append(X[i:i + window_size])
            labels.append(int(y_enc[i + window_size - 1]))
        self.samples = np.stack(samples, axis=0)
        self.labels = np.asarray(labels, dtype=np.int64)
        print(f'windows={len(self)} dims={self.device_dims} step={step} '
              f'classes={sorted(np.unique(self.labels).tolist())}')

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return torch.FloatTensor(self.samples[idx]), torch.tensor(self.labels[idx], dtype=torch.long)


class SimpleMamba(nn.Module):
    def __init__(self, d_model, d_hidden, kernel=4):
        super().__init__()
        self.conv = nn.Conv1d(d_model, d_hidden, kernel_size=kernel, padding=kernel - 1)
        self.gate = nn.Conv1d(d_model, d_hidden, kernel_size=kernel, padding=kernel - 1)
        self.out_proj = nn.Linear(d_hidden, d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        x_t, L = x.transpose(1, 2), x.size(1)
        h = (self.conv(x_t)[:, :, :L] * torch.sigmoid(self.gate(x_t)[:, :, :L])).transpose(1, 2)
        return self.norm(self.out_proj(h) + x)


class EMA:
    def __init__(self, model, decay=0.995):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}

    @torch.no_grad()
    def update(self, model):
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point:
                self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1 - self.decay)
            else:
                self.shadow[k] = v.detach().clone()

    def copy_to(self, model):
        model.load_state_dict(self.shadow, strict=True)


def stratified_timegroup_split(df):
    df = df.copy()
    df['time_group'] = df.index // 2000
    train_indices, val_indices, test_indices = [], [], []
    for _, group in df.groupby('time_group'):
        if len(group) < 3:
            train_indices.extend(group.index.tolist())
            continue
        try:
            sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
            tr_val_idx, test_idx = next(sss.split(group, group['label_encoded']))
        except ValueError:
            tr_val_idx = np.arange(len(group))
            test_idx = np.array([], dtype=int)
        if len(tr_val_idx) > 1:
            try:
                sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
                sub = group.iloc[tr_val_idx]
                train_idx, val_idx = next(sss2.split(sub, sub['label_encoded']))
                train_indices.extend(group.index[tr_val_idx][train_idx].tolist())
                val_indices.extend(group.index[tr_val_idx][val_idx].tolist())
            except ValueError:
                train_indices.extend(group.index[tr_val_idx].tolist())
        else:
            train_indices.extend(group.index[tr_val_idx].tolist())
        if len(test_idx):
            test_indices.extend(group.index[test_idx].tolist())
    return (
        df.loc[train_indices].sort_index().reset_index(drop=True),
        df.loc[val_indices].sort_index().reset_index(drop=True),
        df.loc[test_indices].sort_index().reset_index(drop=True),
    )


def focal_loss(logits, labels, alpha=0.25, gamma=2.0, weight=None):
    ce = F.cross_entropy(logits.float(), labels, reduction='none',
                         weight=weight, label_smoothing=0.015)
    pt = torch.exp(-ce.detach())
    return (alpha * (1 - pt).clamp(0, 1) ** gamma * ce).mean()


def soft_f1_loss(logits, y, n_cls, eps=1e-6):
    p = F.softmax(logits.float(), 1)
    t = F.one_hot(y, n_cls).float()
    tp = (p * t).sum(0)
    fp = (p * (1 - t)).sum(0)
    fn = ((1 - p) * t).sum(0)
    f1 = (2 * tp + eps) / (2 * tp + fp + fn + eps)
    return 1 - f1.mean()


@torch.no_grad()
def collect_logits(model, loader, device):
    model.eval()
    logits_all, trues = [], []
    for x, y in loader:
        x = x.to(device)
        logits_all.append(model(x).float().cpu())
        trues.append(y)
    return torch.cat(logits_all, 0), torch.cat(trues, 0)


def tune_logit_adjustment(logits, y_true, class_prior, labels):
    log_prior = torch.log(torch.clamp(class_prior, min=1e-6))
    best_tau, best_f1 = 0.0, -1.0
    y_np = y_true.numpy()
    for tau in np.linspace(0.0, 1.5, 16):
        pred = (logits - tau * log_prior).argmax(1).numpy()
        f1 = f1_score(y_np, pred, average='macro', labels=labels, zero_division=0)
        if f1 > best_f1:
            best_f1, best_tau = float(f1), float(tau)
    return best_tau, best_f1


@torch.no_grad()
def evaluate(model, loader, device, label_encoder, desc='Eval',
             class_prior=None, adj_tau=0.0):
    model.eval()
    preds, trues = [], []
    log_prior = None
    if class_prior is not None and adj_tau != 0:
        log_prior = torch.log(torch.clamp(class_prior, min=1e-6)).to(device)
    for x, y in tqdm(loader, desc=desc, leave=False):
        x, y = x.to(device), y.to(device)
        logits = model(x).float()
        if log_prior is not None:
            logits = logits - adj_tau * log_prior
        pred = logits.argmax(1)
        preds.extend(pred.cpu().numpy())
        trues.extend(y.cpu().numpy())
    all_labels = list(range(len(label_encoder.classes_)))
    names = list(label_encoder.classes_)
    report = classification_report(
        trues, preds, labels=all_labels, target_names=names, digits=4, zero_division=0)
    f1 = f1_score(trues, preds, average='macro', labels=all_labels, zero_division=0)
    acc = float(accuracy_score(trues, preds))
    return {
        'report': report, 'macro_f1': float(f1), 'accuracy': acc,
        'y_true': np.asarray(trues), 'y_pred': np.asarray(preds),
    }


def plot_confusion_matrix(y_true, y_pred, class_names, save_path):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))
    cm_norm = cm.astype('float') / np.maximum(cm.sum(axis=1, keepdims=True), 1) * 100
    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=axes[0])
    axes[0].set_title('Confusion Matrix (Count)')
    axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')
    sns.heatmap(cm_norm, annot=True, fmt='.1f', cmap='RdYlGn',
                xticklabels=class_names, yticklabels=class_names,
                ax=axes[1], vmin=0, vmax=100)
    axes[1].set_title('Confusion Matrix (Normalized %)')
    axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')
    plt.tight_layout()
    plt.savefig(save_path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f'confusion matrix saved: {save_path}')


class InnovativeGATWithMambaUltraLite(nn.Module):
    
    def __init__(self, device_dims, d_model=16, d_hidden=32, num_classes=8,
                 window_size=128, gat_heads=1, dropout=0.1):
        super().__init__()
        self.num_devices = len(device_dims)
        self.device_dims = device_dims
        self.d_model = d_model
        self.device_projectors = nn.ModuleList([
            nn.Sequential(nn.Linear(dim, d_model), nn.LayerNorm(d_model), nn.GELU())
            for dim in device_dims
        ])
        self.share_logit = nn.Parameter(torch.tensor(-2.0))
        self.act_temp = nn.Parameter(torch.tensor(1.0))
        prior_weight = torch.zeros((self.num_devices, self.num_devices))
        for i, j in [(4, 5), (5, 4), (2, 3), (3, 2), (4, 6), (6, 4)]:
            if i < self.num_devices and j < self.num_devices:
                prior_weight[i, j] = 1.0
        for i, j in [(0, 1), (1, 0), (0, 2), (2, 0)]:
            if i < self.num_devices and j < self.num_devices:
                prior_weight[i, j] = 0.2
        prior_weight[prior_weight == 0] = 0.1
        self.register_buffer('prior_weight', prior_weight)
        self.dynamic_prior_gate = nn.Sequential(
            nn.Linear(d_model * 2, d_model), nn.GELU(), nn.Linear(d_model, 1), nn.Sigmoid())
        self.gat1 = GATv2Conv(d_model, d_model, heads=gat_heads, concat=False,
                              dropout=dropout, add_self_loops=False, edge_dim=1)
        self.n1 = nn.LayerNorm(d_model)
        mamba_dim = d_model * self.num_devices
        self.mamba = SimpleMamba(mamba_dim, d_hidden, kernel=3)
        self.attn = nn.Linear(mamba_dim, 1)
        self.classifier = nn.Sequential(
            nn.LayerNorm(mamba_dim * 2),
            nn.Linear(mamba_dim * 2, num_classes))

    def calibrate_cross_protocol(self, device_feats):
       
        act = device_feats.norm(dim=-1, keepdim=True)
        temp = self.act_temp.clamp(0.2, 5.0)
        w = torch.softmax(act / temp, dim=2)
        shared = (device_feats * w).sum(dim=2, keepdim=True)
        gamma = torch.sigmoid(self.share_logit)
        return (1.0 - gamma) * device_feats + gamma * shared

    def compute_dynamic_edges(self, pooled):
        B, N, D = pooled.shape
        device = pooled.device
        ei = torch.tensor(
            [[i, j] for i in range(N) for j in range(N) if i != j],
            dtype=torch.long, device=device).t().contiguous()
        normed = F.normalize(pooled, p=2, dim=-1)
        sim = torch.matmul(normed, normed.transpose(1, 2))
        dyn = torch.stack([sim[:, i, j] for i, j in zip(ei[0], ei[1])], dim=-1)
        fi, fj = pooled[:, ei[0]], pooled[:, ei[1]]
        alpha = self.dynamic_prior_gate(torch.cat([fi, fj], -1)).squeeze(-1)
        prior = self.prior_weight[ei[0], ei[1]].view(1, -1)
        agree = ((fi * fj).sum(-1) / (fi.norm(dim=-1) * fj.norm(dim=-1) + 1e-6)).clamp(0, 1)
        ew = ((alpha * prior + (1 - alpha) * dyn) * (0.5 + 0.5 * agree)).reshape(-1, 1)
        off = torch.arange(B, device=device) * N
        eib = (ei.unsqueeze(1) + off.view(1, -1, 1)).reshape(2, -1)
        return eib, ew

    def forward(self, x):
        B, T, _ = x.shape
        device_feats, split_idx = [], 0
        for proj, dim in zip(self.device_projectors, self.device_dims):
            device_feats.append(proj(x[:, :, split_idx:split_idx + dim]))
            split_idx += dim
        device_feats = torch.stack(device_feats, dim=2)
        device_feats = self.calibrate_cross_protocol(device_feats)
        pooled = device_feats.mean(1)
        edge_index, edge_weights = self.compute_dynamic_edges(pooled)
        h = pooled.reshape(-1, self.d_model)
        h = self.n1(h + self.gat1(h, edge_index, edge_attr=edge_weights))
        out_seq = device_feats + h.reshape(B, 1, self.num_devices, self.d_model)
        fused = self.mamba(out_seq.reshape(B, T, -1))
        w = torch.softmax(self.attn(fused).squeeze(-1), dim=1)
        feat = torch.cat([(fused * w.unsqueeze(-1)).sum(1), fused.mean(1)], dim=-1)
        return self.classifier(feat)


def kd_loss(student_logits, teacher_logits, T=2.0):
   
    s = F.log_softmax(student_logits.float() / T, dim=1)
    t = F.softmax(teacher_logits.float() / T, dim=1)
    return F.kl_div(s, t, reduction='batchmean') * (T * T)


def train_lite_epoch(student, loader, optimizer, device, n_cls,
                     class_weight, ema, lam_kd=0.65, lam_f1=0.15, kd_temp=1.0):
   
    student.train()
    total, all_preds, all_labels = 0.0, [], []
    pbar = tqdm(loader, desc='UltraTrain', leave=False)
    for x, y in pbar:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        with torch.no_grad():
            bak = {k: v.detach().clone() for k, v in student.state_dict().items()}
            ema.copy_to(student)
            student.eval()
            t_logits = student(x)
            student.train()
            student.load_state_dict(bak, strict=True)
        s_logits = student(x)
        hard = focal_loss(s_logits, y, weight=class_weight) + lam_f1 * soft_f1_loss(s_logits, y, n_cls)
        loss = (1 - lam_kd) * hard + lam_kd * kd_loss(s_logits, t_logits, T=kd_temp) if lam_kd > 0 else hard
        if torch.isnan(loss):
            continue
        loss.backward()
        torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
        optimizer.step()
        ema.update(student)
        bs = x.size(0)
        total += loss.item() * bs
        all_preds.extend(s_logits.argmax(1).detach().cpu().numpy())
        all_labels.extend(y.cpu().numpy())
        pbar.set_postfix(loss=float(loss.item()))
    n = max(len(loader.dataset), 1)
    f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return total / n, f1


WINDOW_SIZE, TRAIN_STEP, EVAL_STEP = 128, 1, 2
CLF_TAG, DROPOUT, SCHEDULER_NAME = 'A', 0.1, 'plateau'
BATCH_SIZE, EPOCHS, LR, PATIENCE = 64, 50, 1e-3, 5
FORCE_FRESH = True  
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'UltraLite (second) device={DEVICE}  MeanTeacher lam_kd={LAM_SELF_KD} ema={EMA_DECAY} T={KD_TEMP}')
print(f'hp: clf={CLF_TAG} lr={LR} dropout={DROPOUT} batch={BATCH_SIZE} '
      f'sch={SCHEDULER_NAME} patience={PATIENCE}')
print(f'data protocol: window={WINDOW_SIZE} train_step={TRAIN_STEP} eval_step={EVAL_STEP}')
print(f'save={LITE_CKPT}')
if FORCE_FRESH and os.path.isfile(LITE_CKPT):
    bak = LITE_CKPT.replace('.pth', '_pre_meanteacher.pth')
    shutil.copy2(LITE_CKPT, bak)
    os.remove(LITE_CKPT)
    print(f'FORCE_FRESH: backed up old ckpt -> {bak}')
if FORCE_FRESH and os.path.isfile(LITE_RESULT):
    bak_r = LITE_RESULT.replace('.txt', '_pre_meanteacher.txt')
    shutil.copy2(LITE_RESULT, bak_r)
    print(f'FORCE_FRESH: backed up old results -> {bak_r}')
assert os.path.isfile(DATA_CSV), f'找不到数据: {DATA_CSV}'

df = pd.read_csv(DATA_CSV, low_memory=False)
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values('timestamp').reset_index(drop=True)
valid_types = df['final_type'].value_counts()[df['final_type'].value_counts() > 10].index
df = df[df['final_type'].isin(valid_types)].copy()
label_encoder = LabelEncoder()
df['label_encoded'] = label_encoder.fit_transform(df['final_type'])
n_cls = len(label_encoder.classes_)
print('classes:', list(label_encoder.classes_))

df_train, df_val, df_test = stratified_timegroup_split(df)
print(f'train/val/test rows: {len(df_train)}/{len(df_val)}/{len(df_test)}')

train_ds = IoTFusionDataset(df_train, window_size=WINDOW_SIZE, step=TRAIN_STEP,
                            fit_scaler=True, label_encoder=label_encoder)
val_ds = IoTFusionDataset(df_val, window_size=WINDOW_SIZE, step=EVAL_STEP,
                          scaler=train_ds.scaler, label_encoder=label_encoder)
test_ds = IoTFusionDataset(df_test, window_size=WINDOW_SIZE, step=EVAL_STEP,
                           scaler=train_ds.scaler, label_encoder=label_encoder)

counts = np.maximum(np.bincount(train_ds.labels, minlength=n_cls).astype(np.float64), 1.0)
sw = 1.0 / np.sqrt(counts[train_ds.labels])
sw = sw / sw.mean()
sampler = torch.utils.data.WeightedRandomSampler(torch.DoubleTensor(sw), len(train_ds), True)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=0, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=0, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=0, pin_memory=True)
cw = np.clip(np.sqrt(counts.sum() / (n_cls * counts)), 0.5, 3.0)
class_weight = torch.tensor(cw, dtype=torch.float32, device=DEVICE)
class_prior = torch.tensor(counts / counts.sum(), dtype=torch.float32)

student = InnovativeGATWithMambaUltraLite(
    device_dims=train_ds.device_dims, d_model=16, d_hidden=32,
    num_classes=n_cls, window_size=WINDOW_SIZE, gat_heads=1, dropout=DROPOUT,
).to(DEVICE)
n_params = sum(p.numel() for p in student.parameters())
print(f'ultralite params={n_params/1e6:.3f}M  | MeanTeacher ON | activity-weighted soft share')

ema = EMA(student, decay=EMA_DECAY)
optimizer = torch.optim.AdamW(student.parameters(), lr=LR, weight_decay=1e-4)
assert SCHEDULER_NAME == 'plateau'
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=2)

best_val_f1, best_epoch, bad = -1.0, 0, 0
start_epoch = 1


def consistency_rampup(epoch, ramp_epochs=RAMP_EPOCHS, max_w=LAM_SELF_KD):
    """Mean Teacher (Tarvainen & Valpola): soft-label weight ramp-up."""
    t = 1.0 if ramp_epochs <= 0 else min(1.0, float(epoch) / float(ramp_epochs))
    return float(max_w * math.exp(-5.0 * (1.0 - t) ** 2))


for epoch in range(start_epoch, EPOCHS + 1):
    lam_now = consistency_rampup(epoch)
    print(f'\n--- UltraLite Epoch {epoch}/{EPOCHS} --- lam_kd={lam_now:.3f}')
    tr_loss, tr_f1 = train_lite_epoch(
        student, train_loader, optimizer, DEVICE, n_cls, class_weight, ema,
        lam_kd=lam_now, lam_f1=LAM_F1, kd_temp=KD_TEMP)
    bak = {k: v.detach().clone() for k, v in student.state_dict().items()}
    ema.copy_to(student)
    val = evaluate(student, val_loader, DEVICE, label_encoder, desc='UltraValid')
    student.load_state_dict(bak)
    print(f'Train Loss {tr_loss:.4f} | Train F1 {tr_f1:.4f} | Val Macro-F1 {val["macro_f1"]:.4f}')
    if val['macro_f1'] > best_val_f1 + 1e-4:
        best_val_f1, best_epoch, bad = val['macro_f1'], epoch, 0
        torch.save({
            'model_state': {k: v.cpu().clone() for k, v in ema.shadow.items()},
            'epoch': best_epoch,
            'val_macro_f1': best_val_f1,
            'classes': list(label_encoder.classes_),
            'device_dims': list(train_ds.device_dims),
            'd_model': 16, 'd_hidden': 32, 'gat_heads': 1,
            'dropout': DROPOUT, 'clf': CLF_TAG, 'lr': LR, 'batch_size': BATCH_SIZE,
            'scheduler': SCHEDULER_NAME, 'patience': PATIENCE,
            'model': 'InnovativeGATWithMambaUltraLite',
            'use_self_kd': True,
            'lam_kd': LAM_SELF_KD,
            'ema_decay': EMA_DECAY,
            'ramp_epochs': RAMP_EPOCHS,
            'kd_temp': KD_TEMP,
            'teacher': 'EMA-MeanTeacher',
            'class_prior': class_prior.cpu(),
        }, LITE_CKPT)
        print(f'  >> saved {LITE_CKPT}  Val Macro-F1={best_val_f1:.4f}')
    else:
        bad += 1
        if bad >= PATIENCE:
            print(f'early stop (patience={PATIENCE})')
            break
    scheduler.step(val['macro_f1'])
    print(f'  lr={optimizer.param_groups[0]["lr"]:.2e}')

ckpt = torch.load(LITE_CKPT, map_location=DEVICE, weights_only=False)
student.load_state_dict(ckpt['model_state'])
val_logits, val_y = collect_logits(student, val_loader, DEVICE)
adj_tau, cal_f1 = tune_logit_adjustment(val_logits, val_y, class_prior, list(range(n_cls)))
ckpt['adj_tau'] = adj_tau
torch.save(ckpt, LITE_CKPT)

val = evaluate(student, val_loader, DEVICE, label_encoder, desc='UltraValid',
               class_prior=class_prior, adj_tau=adj_tau)
test = evaluate(student, test_loader, DEVICE, label_encoder, desc='UltraTest',
                class_prior=class_prior, adj_tau=adj_tau)
print('\n' + '=' * 60)
print('GAT-Mamba UltraLite (second, Mean-Teacher self-distillation)')
print(f'clf={CLF_TAG} lr={LR} drop={DROPOUT} batch={BATCH_SIZE} sch={SCHEDULER_NAME} pat={PATIENCE}')
print(f'Best Epoch {best_epoch} | adj_tau={adj_tau:.3f} | params={n_params/1e6:.3f}M')
print(f'Val Macro-F1 {val["macro_f1"]:.4f} | Test Macro-F1 {test["macro_f1"]:.4f} | Acc {test["accuracy"]:.4f}')
print('=' * 60)
print(test['report'])
plot_confusion_matrix(test['y_true'], test['y_pred'], list(label_encoder.classes_), save_path=LITE_CM)
with open(LITE_RESULT, 'w', encoding='utf-8') as f:
    f.write('model=InnovativeGATWithMambaUltraLite\n')
    f.write(f'params_M={n_params/1e6:.6f}\n')
    f.write(f'data=iot_fusion_weather_all.csv window={WINDOW_SIZE} '
            f'step={TRAIN_STEP}/{EVAL_STEP}\n')
    f.write(f'use_self_kd=True\nlam_kd={LAM_SELF_KD}\nema_decay={EMA_DECAY}\n'
            f'ramp_epochs={RAMP_EPOCHS}\nkd_temp={KD_TEMP}\n')
    f.write(f'clf={CLF_TAG}\nlr={LR}\ndropout={DROPOUT}\nbatch_size={BATCH_SIZE}\n'
            f'scheduler={SCHEDULER_NAME}\npatience={PATIENCE}\n')
    f.write(f'best_epoch={best_epoch}\nadj_tau={adj_tau:.6f}\n')
    f.write(f'val_macro_f1={val["macro_f1"]:.6f}\n')
    f.write(f'test_macro_f1={test["macro_f1"]:.6f}\n')
    f.write(f'test_accuracy={test["accuracy"]:.6f}\n')
    f.write('note=Mean-Teacher self-KD + consistency ramp-up\n\n')
    f.write('===== VAL =====\n' + val['report'] + '\n')
    f.write('===== TEST =====\n' + test['report'] + '\n')
print(f'saved: {LITE_CKPT} | {LITE_RESULT}')
print('main model untouched: best_innovative_model.pth')


## 消融实验（相对两个创新点）

协议与主实验对齐：`iot_fusion_weather_all.csv`，`window=128`，`train_step=1`，`eval_step=2`，`batch=64`，`lr=1e-3`，`patience=5`，`dropout=0.1`，`ReduceLROnPlateau`。

| 组 | 配置 | 先验动态门控 GAT | SimpleMamba + 自蒸馏 |
|----|------|------------------|----------------------|
| A0 | 原始特征 mean + 线性分类 | ✗ | ✗ |
| A1 | 仅先验动态门控 GAT | ✓ | ✗ |
| A2 | SimpleMamba + Mean-Teacher | ✗ | ✓ |

输出：`best_ablation_A0_none_*`、`best_ablation_A1_dyn_prior_gat_*`、`best_ablation_A2_mamba_kd_*`、`ablation_summary.txt`


In [ ]:
import matplotlib
matplotlib.use('Agg')


import os
import math
import tempfile
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, f1_score, accuracy_score, confusion_matrix
from sklearn.model_selection import StratifiedShuffleSplit
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from torch_geometric.nn import GATv2Conv

_CANDS = [
    r'E:\apt\YES\2',
    os.path.abspath(os.getcwd()),
    os.path.dirname(os.path.abspath(os.getcwd())),
]
_ROOT = None
for _cand in _CANDS:
    if os.path.isfile(os.path.join(_cand, 'iot_fusion_weather_all.csv')):
        _ROOT = _cand
        os.chdir(_ROOT)
        break
if _ROOT is None:
    raise FileNotFoundError('数据集不存在: ' + ' | '.join(_CANDS))
DATA_CSV = os.path.join(_ROOT, 'iot_fusion_weather_all.csv')
_TMP = os.path.join(_ROOT, '_tmp')
os.makedirs(_TMP, exist_ok=True)
for _k in ('TEMP', 'TMP', 'TMPDIR'):
    os.environ[_k] = _TMP
os.environ['TORCH_HOME'] = os.path.join(_TMP, 'torch')
os.environ['MPLCONFIGDIR'] = os.path.join(_TMP, 'matplotlib')
os.makedirs(os.environ['TORCH_HOME'], exist_ok=True)
os.makedirs(os.environ['MPLCONFIGDIR'], exist_ok=True)
tempfile.tempdir = _TMP

SUMMARY_TXT = os.path.join(_ROOT, 'ablation_summary.txt')
print(f'输出目录(当前路径)={_ROOT}')
print(f'数据文件={DATA_CSV}')

WINDOW_SIZE, TRAIN_STEP, EVAL_STEP = 128, 1, 2
BATCH_SIZE, EPOCHS, LR, PATIENCE = 64, 50, 1e-3, 5
DROPOUT, D_MODEL, D_HIDDEN, GAT_HEADS = 0.1, 16, 32, 1
LAM_F1 = 0.15
LAM_SELF_KD = 0.65
EMA_DECAY = 0.999
RAMP_EPOCHS = 10
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

DEVICE_PREFIXES = ['weather', 'modbus', 'thermostat', 'fridge',
                   'garage_door', 'gps', 'motion_light']

ABLATION_CONFIGS = [
    dict(
        tag='A0_none', name='A0_纯基线',
        use_gat=False, use_mamba=False, use_mean_teacher=False, pool='mean',
        ckpt='best_ablation_A0_none.pth',
        result='best_ablation_A0_none_results.txt',
        cm='cm_A0_none.png'
    ),
    dict(
        tag='A1_dyn_prior_gat', name='A1_先验动态门控GAT',
        use_gat=True, use_mamba=False, use_mean_teacher=False, pool='attn_mean',
        ckpt='best_ablation_A1_dyn_prior_gat.pth',
        result='best_ablation_A1_dyn_prior_gat_results.txt',
        cm='cm_A1_dyn_prior_gat.png'
    ),
    dict(
        tag='A2_mamba_kd', name='A2_SimpleMamba+自蒸馏',
        use_gat=False, use_mamba=True, use_mean_teacher=True, pool='attn_mean',
        ckpt='best_ablation_A2_mamba_kd.pth',
        result='best_ablation_A2_mamba_kd_results.txt',
        cm='cm_A2_mamba_kd.png'
    ),
]
RETRAIN_TAGS = {'A0_none', 'A1_dyn_prior_gat', 'A2_mamba_kd'}


class IoTFusionDataset(Dataset):
    def __init__(self, df, window_size=128, step=1, scaler=None,
                 fit_scaler=False, label_encoder=None):
        self.window_size = window_size
        self.step = step
        self.label_encoder = label_encoder
        self.devices = DEVICE_PREFIXES
        self.num_devices = len(DEVICE_PREFIXES)
        self.device_feat_cols = {}
        self.device_dims = []
        self.all_feat_cols = []
        for dev in DEVICE_PREFIXES:
            cols = [c for c in df.columns if c.startswith(dev + '_') and
                    not c.endswith('_label') and not c.endswith('_type')]
            self.device_feat_cols[dev] = cols
            self.device_dims.append(len(cols))
            self.all_feat_cols.extend(cols)

        X = df[self.all_feat_cols].values.astype(np.float32)
        y_enc = df['label_encoded'].values.astype(np.int64)
        if scaler is None and fit_scaler:
            self.scaler = StandardScaler()
            X = self.scaler.fit_transform(X)
        elif scaler is not None:
            self.scaler = scaler
            X = self.scaler.transform(X)
        else:
            self.scaler = None
        X = np.nan_to_num(np.clip(X, -8, 8), nan=0.0).astype(np.float32)

        samples, labels = [], []
        for i in range(0, len(X) - window_size + 1, step):
            samples.append(X[i:i + window_size])
            labels.append(int(y_enc[i + window_size - 1]))
        self.samples = np.stack(samples, axis=0)
        self.labels = np.asarray(labels, dtype=np.int64)
        print(f'windows={len(self)} dims={self.device_dims} step={step} '
              f'classes={sorted(np.unique(self.labels).tolist())}')

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return torch.FloatTensor(self.samples[idx]), torch.tensor(self.labels[idx], dtype=torch.long)


class SimpleMamba(nn.Module):
    def __init__(self, d_model, d_hidden, kernel=3):
        super().__init__()
        self.conv = nn.Conv1d(d_model, d_hidden, kernel_size=kernel, padding=kernel - 1)
        self.gate = nn.Conv1d(d_model, d_hidden, kernel_size=kernel, padding=kernel - 1)
        self.out_proj = nn.Linear(d_hidden, d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        x_t, L = x.transpose(1, 2), x.size(1)
        h = (self.conv(x_t)[:, :, :L] * torch.sigmoid(self.gate(x_t)[:, :, :L])).transpose(1, 2)
        return self.norm(self.out_proj(h) + x)


class EMA:
    def __init__(self, model, decay=EMA_DECAY):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}

    @torch.no_grad()
    def update(self, model):
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point:
                self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1 - self.decay)
            else:
                self.shadow[k] = v.detach().clone()

    def copy_to(self, model):
        model.load_state_dict(self.shadow, strict=True)


def stratified_timegroup_split(df):
    df = df.copy()
    df['time_group'] = df.index // 2000
    train_indices, val_indices, test_indices = [], [], []
    for _, group in df.groupby('time_group'):
        if len(group) < 3:
            train_indices.extend(group.index.tolist())
            continue
        try:
            sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
            tr_val_idx, test_idx = next(sss.split(group, group['label_encoded']))
        except ValueError:
            tr_val_idx = np.arange(len(group))
            test_idx = np.array([], dtype=int)
        if len(tr_val_idx) > 1:
            try:
                sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
                sub = group.iloc[tr_val_idx]
                train_idx, val_idx = next(sss2.split(sub, sub['label_encoded']))
                train_indices.extend(group.index[tr_val_idx][train_idx].tolist())
                val_indices.extend(group.index[tr_val_idx][val_idx].tolist())
            except ValueError:
                train_indices.extend(group.index[tr_val_idx].tolist())
        else:
            train_indices.extend(group.index[tr_val_idx].tolist())
        if len(test_idx):
            test_indices.extend(group.index[test_idx].tolist())
    return (
        df.loc[train_indices].sort_index().reset_index(drop=True),
        df.loc[val_indices].sort_index().reset_index(drop=True),
        df.loc[test_indices].sort_index().reset_index(drop=True),
    )


def focal_loss(logits, labels, alpha=0.25, gamma=2.0, weight=None):
    ce = F.cross_entropy(logits.float(), labels, reduction='none',
                         weight=weight, label_smoothing=0.015)
    pt = torch.exp(-ce.detach())
    return (alpha * (1 - pt).clamp(0, 1) ** gamma * ce).mean()


def soft_f1_loss(logits, y, n_cls, eps=1e-6):
    p = F.softmax(logits.float(), 1)
    t = F.one_hot(y, n_cls).float()
    tp = (p * t).sum(0)
    fp = (p * (1 - t)).sum(0)
    fn = ((1 - p) * t).sum(0)
    f1 = (2 * tp + eps) / (2 * tp + fp + fn + eps)
    return 1 - f1.mean()


def kd_loss(student_logits, teacher_logits, T=2.0):
    s = F.log_softmax(student_logits.float() / T, dim=1)
    t = F.softmax(teacher_logits.float() / T, dim=1)
    return F.kl_div(s, t, reduction='batchmean') * (T * T)


def consistency_rampup(epoch, ramp_epochs=RAMP_EPOCHS, max_w=LAM_SELF_KD):
    t = 1.0 if ramp_epochs <= 0 else min(1.0, float(epoch) / float(ramp_epochs))
    return float(max_w * math.exp(-5.0 * (1.0 - t) ** 2))


@torch.no_grad()
def evaluate(model, loader, device, label_encoder, desc='Eval'):
    model.eval()
    preds, trues = [], []
    for x, y in tqdm(loader, desc=desc, leave=False):
        x = x.to(device)
        logits = model(x).float()
        preds.extend(logits.argmax(1).cpu().numpy())
        trues.extend(y.numpy())
    y_true = np.asarray(trues)
    y_pred = np.asarray(preds)
    all_labels = list(range(len(label_encoder.classes_)))
    names = list(label_encoder.classes_)
    report = classification_report(
        trues, preds, labels=all_labels, target_names=names, digits=4, zero_division=0)
    f1 = f1_score(y_true, y_pred, average='macro', labels=all_labels, zero_division=0)
    acc = float(accuracy_score(y_true, y_pred))
    return {
        'report': report, 'macro_f1': float(f1), 'accuracy': acc,
        'y_true': y_true, 'y_pred': y_pred,
    }


def plot_confusion_matrix(y_true, y_pred, class_names, save_path):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))
    cm_norm = cm.astype('float') / np.maximum(cm.sum(axis=1, keepdims=True), 1) * 100
    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=axes[0])
    axes[0].set_title('Confusion Matrix (Count)')
    axes[0].set_xlabel('Predicted')
    axes[0].set_ylabel('True')
    sns.heatmap(cm_norm, annot=True, fmt='.1f', cmap='RdYlGn',
                xticklabels=class_names, yticklabels=class_names,
                ax=axes[1], vmin=0, vmax=100)
    axes[1].set_title('Confusion Matrix (Normalized %)')
    axes[1].set_xlabel('Predicted')
    axes[1].set_ylabel('True')
    plt.tight_layout()
    plt.savefig(save_path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f'confusion matrix saved: {save_path}')


class BaselineEncoder(nn.Module):
    
    def __init__(self, device_dims, d_model=D_MODEL, num_classes=8, dropout=DROPOUT,
                 pool='mean'):
        super().__init__()
        self.num_devices = len(device_dims)
        self.device_dims = device_dims
        self.d_model = d_model
        self.pool = pool
        feat_dim = int(sum(device_dims))
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(feat_dim, num_classes)

    def forward(self, x):
        feat = x.mean(dim=1) if self.pool != 'last' else x[:, -1]
        return self.classifier(self.dropout(feat))


class DynamicPriorGATEncoder(nn.Module):
    
    def __init__(self, device_dims, d_model=D_MODEL, num_classes=8,
                 gat_heads=GAT_HEADS, dropout=DROPOUT):
        super().__init__()
        self.num_devices = len(device_dims)
        self.device_dims = device_dims
        self.d_model = d_model
        self.device_projectors = nn.ModuleList([
            nn.Sequential(nn.Linear(dim, d_model), nn.LayerNorm(d_model), nn.GELU())
            for dim in device_dims
        ])
        prior_weight = torch.zeros((self.num_devices, self.num_devices))
        for i, j in [(4, 5), (5, 4), (2, 3), (3, 2), (4, 6), (6, 4)]:
            if i < self.num_devices and j < self.num_devices:
                prior_weight[i, j] = 1.0
        for i, j in [(0, 1), (1, 0), (0, 2), (2, 0)]:
            if i < self.num_devices and j < self.num_devices:
                prior_weight[i, j] = 0.2
        prior_weight[prior_weight == 0] = 0.1
        self.register_buffer('prior_weight', prior_weight)
        self.dynamic_prior_gate = nn.Sequential(
            nn.Linear(d_model * 2, d_model), nn.GELU(), nn.Linear(d_model, 1), nn.Sigmoid())
        self.gat1 = GATv2Conv(d_model, d_model, heads=gat_heads, concat=False,
                              dropout=dropout, add_self_loops=False, edge_dim=1)
        self.n1 = nn.LayerNorm(d_model)
        out_dim = d_model * self.num_devices
        self.attn = nn.Linear(out_dim, 1)
        self.classifier = nn.Sequential(
            nn.LayerNorm(out_dim * 2),
            nn.Linear(out_dim * 2, num_classes))

    def compute_dynamic_edges(self, pooled):
        B, N, D = pooled.shape
        device = pooled.device
        ei = torch.tensor(
            [[i, j] for i in range(N) for j in range(N) if i != j],
            dtype=torch.long, device=device).t().contiguous()
        normed = F.normalize(pooled, p=2, dim=-1)
        sim = torch.matmul(normed, normed.transpose(1, 2))
        dyn = torch.stack([sim[:, i, j] for i, j in zip(ei[0], ei[1])], dim=-1)
        fi, fj = pooled[:, ei[0]], pooled[:, ei[1]]
        alpha = self.dynamic_prior_gate(torch.cat([fi, fj], -1)).squeeze(-1)
        prior = self.prior_weight[ei[0], ei[1]].view(1, -1)
        agree = ((fi * fj).sum(-1) / (fi.norm(dim=-1) * fj.norm(dim=-1) + 1e-6)).clamp(0, 1)
        ew = ((alpha * prior + (1 - alpha) * dyn) * (0.5 + 0.5 * agree)).reshape(-1, 1)
        off = torch.arange(B, device=device) * N
        eib = (ei.unsqueeze(1) + off.view(1, -1, 1)).reshape(2, -1)
        return eib, ew

    def forward(self, x):
        B, T, _ = x.shape
        device_feats, split_idx = [], 0
        for proj, dim in zip(self.device_projectors, self.device_dims):
            device_feats.append(proj(x[:, :, split_idx:split_idx + dim]))
            split_idx += dim
        device_feats = torch.stack(device_feats, dim=2)
        pooled = device_feats.mean(1)
        edge_index, edge_weights = self.compute_dynamic_edges(pooled)
        h = pooled.reshape(-1, self.d_model)
        h = self.n1(h + self.gat1(h, edge_index, edge_attr=edge_weights))
        out_seq = device_feats + h.reshape(B, 1, self.num_devices, self.d_model)
        fused = out_seq.reshape(B, T, -1)
        w = torch.softmax(self.attn(fused).squeeze(-1), dim=1)
        feat = torch.cat([(fused * w.unsqueeze(-1)).sum(1), fused.mean(1)], dim=-1)
        return self.classifier(feat)


class SimpleMambaEncoder(nn.Module):
    
    def __init__(self, device_dims, d_model=D_MODEL, d_hidden=D_HIDDEN,
                 num_classes=8, dropout=DROPOUT):
        super().__init__()
        self.num_devices = len(device_dims)
        self.device_dims = device_dims
        self.d_model = d_model
        self.device_projectors = nn.ModuleList([
            nn.Sequential(nn.Linear(dim, d_model), nn.LayerNorm(d_model), nn.GELU())
            for dim in device_dims
        ])
        self.dropout = nn.Dropout(dropout)
        mamba_dim = d_model * self.num_devices
        self.mamba = SimpleMamba(mamba_dim, d_hidden, kernel=3)
        self.attn = nn.Linear(mamba_dim, 1)
        self.classifier = nn.Sequential(
            nn.LayerNorm(mamba_dim * 2),
            nn.Linear(mamba_dim * 2, num_classes))

    def forward(self, x):
        B, T, _ = x.shape
        device_feats, split_idx = [], 0
        for proj, dim in zip(self.device_projectors, self.device_dims):
            device_feats.append(proj(x[:, :, split_idx:split_idx + dim]))
            split_idx += dim
        fused = torch.cat(device_feats, dim=-1)
        fused = self.dropout(fused)
        fused = self.mamba(fused)
        w = torch.softmax(self.attn(fused).squeeze(-1), dim=1)
        feat = torch.cat([(fused * w.unsqueeze(-1)).sum(1), fused.mean(1)], dim=-1)
        return self.classifier(feat)


def build_ablation_model(cfg, device_dims, n_cls):
    if cfg['use_gat']:
        return DynamicPriorGATEncoder(
            device_dims=device_dims, d_model=D_MODEL, num_classes=n_cls,
            gat_heads=GAT_HEADS, dropout=DROPOUT
        )
    if cfg['use_mamba']:
        return SimpleMambaEncoder(
            device_dims=device_dims, d_model=D_MODEL, d_hidden=D_HIDDEN,
            num_classes=n_cls, dropout=DROPOUT
        )
    return BaselineEncoder(
        device_dims=device_dims, d_model=D_MODEL, num_classes=n_cls,
        dropout=DROPOUT, pool=cfg.get('pool', 'mean'),
    )


def train_one_epoch(model, loader, optimizer, device, n_cls, class_weight,
                    ema=None, use_kd=False, lam_kd=0.0, lam_f1=LAM_F1):
    model.train()
    total_loss, all_preds, all_labels = 0.0, [], []
    pbar = tqdm(loader, desc='Train', leave=False)
    for x, y in pbar:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        t_logits = None
        if use_kd and ema is not None:
            with torch.no_grad():
                bak_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
                ema.copy_to(model)
                model.eval()
                t_logits = model(x)
                model.train()
                model.load_state_dict(bak_state, strict=True)
        s_logits = model(x)
        hard_loss = focal_loss(s_logits, y, weight=class_weight) + lam_f1 * soft_f1_loss(s_logits, y, n_cls)
        if use_kd and t_logits is not None and lam_kd > 0:
            loss = (1 - lam_kd) * hard_loss + lam_kd * kd_loss(s_logits, t_logits)
        else:
            loss = hard_loss
        if torch.isnan(loss):
            continue
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        if ema is not None:
            ema.update(model)
        bs = x.size(0)
        total_loss += loss.item() * bs
        all_preds.extend(s_logits.argmax(1).detach().cpu().numpy())
        all_labels.extend(y.cpu().numpy())
        pbar.set_postfix(loss=float(loss.item()))
    n = max(len(loader.dataset), 1)
    f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return total_loss / n, f1


def run_single_ablation(cfg, train_ds, val_loader, test_loader, label_encoder,
                        class_weight, train_loader):
    n_cls = len(label_encoder.classes_)
    model = build_ablation_model(cfg, train_ds.device_dims, n_cls).to(DEVICE)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\n{'='*70}")
    print(f"[{cfg['name']}] GAT={cfg['use_gat']}  Mamba={cfg['use_mamba']}  "
          f"自蒸馏={cfg['use_mean_teacher']}  pool={cfg.get('pool')}  "
          f"参数量={n_params/1e6:.3f}M")
    print('='*70)

    ema = EMA(model, decay=EMA_DECAY) if cfg['use_mean_teacher'] else None
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=2, min_lr=1e-6)

    ckpt_path = os.path.join(_ROOT, cfg['ckpt'])
    result_path = os.path.join(_ROOT, cfg['result'])
    cm_path = os.path.join(_ROOT, cfg['cm'])
    best_f1, best_epoch, bad = -1.0, 0, 0

    for epoch in range(1, EPOCHS + 1):
        lam_kd = consistency_rampup(epoch) if cfg['use_mean_teacher'] else 0.0
        print(f"\n--- Epoch {epoch}/{EPOCHS}  lam_kd={lam_kd:.3f} ---")
        tr_loss, tr_f1 = train_one_epoch(
            model, train_loader, optimizer, DEVICE, n_cls, class_weight,
            ema=ema, use_kd=cfg['use_mean_teacher'], lam_kd=lam_kd
        )
        if ema is not None:
            bak = {k: v.detach().clone() for k, v in model.state_dict().items()}
            ema.copy_to(model)
            val = evaluate(model, val_loader, DEVICE, label_encoder, desc='Valid')
            model.load_state_dict(bak)
            save_state = {k: v.cpu().clone() for k, v in ema.shadow.items()}
        else:
            val = evaluate(model, val_loader, DEVICE, label_encoder, desc='Valid')
            save_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        print(f'Train Loss={tr_loss:.4f}  Train F1={tr_f1:.4f}  Val Macro-F1={val["macro_f1"]:.4f}')
        if val['macro_f1'] > best_f1 + 1e-4:
            best_f1, best_epoch, bad = val['macro_f1'], epoch, 0
            torch.save({
                'model_state': save_state,
                'epoch': best_epoch,
                'val_macro_f1': best_f1,
                'classes': list(label_encoder.classes_),
                'device_dims': list(train_ds.device_dims),
                'ablation': cfg['tag'],
                'use_gat': cfg['use_gat'],
                'use_mamba': cfg['use_mamba'],
                'use_mean_teacher': cfg['use_mean_teacher'],
                'pool': cfg.get('pool'),
                'd_model': D_MODEL, 'd_hidden': D_HIDDEN, 'gat_heads': GAT_HEADS,
                'dropout': DROPOUT, 'lr': LR, 'batch_size': BATCH_SIZE,
            }, ckpt_path)
            print(f'  >> 最优模型已保存，Val Macro-F1={best_f1:.4f}')
        else:
            bad += 1
            if bad >= PATIENCE:
                print(f'早停触发（patience={PATIENCE}）')
                break
        scheduler.step(val['macro_f1'])
        print(f'  lr={optimizer.param_groups[0]["lr"]:.2e}')

    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt['model_state'], strict=True)
    test = evaluate(model, test_loader, DEVICE, label_encoder, desc='Test')
    val = evaluate(model, val_loader, DEVICE, label_encoder, desc='Valid')

    with open(result_path, 'w', encoding='utf-8') as f:
        f.write(f"消融组: {cfg['name']}\n")
        f.write(f"GAT: {cfg['use_gat']}\n")
        f.write(f"Mamba: {cfg['use_mamba']}\n")
        f.write(f"Mean-Teacher自蒸馏: {cfg['use_mean_teacher']}\n")
        f.write(f"pool: {cfg.get('pool')}\n")
        f.write(f"参数量(M): {n_params/1e6:.6f}\n")
        f.write(f"最优轮次: {best_epoch}\n")
        f.write(f"Val Macro-F1: {val['macro_f1']:.6f}\n")
        f.write(f"Test Macro-F1: {test['macro_f1']:.6f}\n")
        f.write(f"Test Accuracy: {test['accuracy']:.6f}\n\n")
        f.write('===== TEST 分类报告 =====\n')
        f.write(test['report'] + '\n')

    plot_confusion_matrix(test['y_true'], test['y_pred'],
                          list(label_encoder.classes_), cm_path)
    print(f"测试完成 | Test Macro-F1={test['macro_f1']:.4f} | Test Acc={test['accuracy']:.4f}")
    print(f'结果已保存: {result_path}')
    return dict(
        name=cfg['name'],
        use_gat=cfg['use_gat'],
        use_mamba=cfg['use_mamba'],
        use_mean_teacher=cfg['use_mean_teacher'],
        params_M=n_params / 1e6,
        best_epoch=best_epoch,
        val_f1=float(val['macro_f1']),
        test_f1=float(test['macro_f1']),
        test_acc=float(test['accuracy'])
    )


def load_saved_ablation(cfg):
    result_path = os.path.join(_ROOT, cfg['result'])
    kv = {}
    with open(result_path, encoding='utf-8') as f:
        for line in f:
            if ':' in line:
                k, v = line.split(':', 1)
                kv[k.strip()] = v.strip()
    print(f"[{cfg['name']}] 复用已有结果: {result_path}")
    use_gat = kv.get('GAT', str(cfg['use_gat']))
    use_mamba = kv.get('Mamba', str(cfg['use_mamba']))
    use_mt = kv.get('Mean-Teacher自蒸馏', str(cfg['use_mean_teacher']))
    return dict(
        name=cfg['name'],
        use_gat=use_gat.lower() == 'true' if isinstance(use_gat, str) else bool(use_gat),
        use_mamba=use_mamba.lower() == 'true' if isinstance(use_mamba, str) else bool(use_mamba),
        use_mean_teacher=use_mt.lower() == 'true' if isinstance(use_mt, str) else bool(use_mt),
        params_M=float(kv['参数量(M)']),
        best_epoch=int(kv['最优轮次']),
        val_f1=float(kv['Val Macro-F1']),
        test_f1=float(kv['Test Macro-F1']),
        test_acc=float(kv['Test Accuracy']),
    )


def main():
    print('=' * 70)
    print('三组消融实验 | 超参对齐设备侧 UltraLite | 结构对齐目录1')
    print(f'设备: {DEVICE} | 数据: {DATA_CSV}')
    print(f'窗口={WINDOW_SIZE} 步长={TRAIN_STEP}/{EVAL_STEP} 批次={BATCH_SIZE} '
          f'学习率={LR} 早停={PATIENCE} dropout={DROPOUT}')
    print('=' * 70)

    df = pd.read_csv(DATA_CSV, low_memory=False)
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df = df.sort_values('timestamp').reset_index(drop=True)
    valid_types = df['final_type'].value_counts()[df['final_type'].value_counts() > 10].index
    df = df[df['final_type'].isin(valid_types)].copy()

    label_encoder = LabelEncoder()
    df['label_encoded'] = label_encoder.fit_transform(df['final_type'])
    n_cls = len(label_encoder.classes_)
    print(f'\n类别数: {n_cls}')
    print(f'类别列表: {list(label_encoder.classes_)}')

    df_train, df_val, df_test = stratified_timegroup_split(df)
    print(f'原始样本数 | 训练:{len(df_train)} 验证:{len(df_val)} 测试:{len(df_test)}')

    train_ds = IoTFusionDataset(df_train, window_size=WINDOW_SIZE, step=TRAIN_STEP,
                                fit_scaler=True, label_encoder=label_encoder)
    val_ds = IoTFusionDataset(df_val, window_size=WINDOW_SIZE, step=EVAL_STEP,
                              scaler=train_ds.scaler, label_encoder=label_encoder)
    test_ds = IoTFusionDataset(df_test, window_size=WINDOW_SIZE, step=EVAL_STEP,
                               scaler=train_ds.scaler, label_encoder=label_encoder)

    counts = np.maximum(np.bincount(train_ds.labels, minlength=n_cls).astype(np.float64), 1.0)
    sw = 1.0 / np.sqrt(counts[train_ds.labels])
    sw = sw / sw.mean()
    sampler = torch.utils.data.WeightedRandomSampler(torch.DoubleTensor(sw), len(train_ds), True)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                              num_workers=0, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=0, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=0, pin_memory=True)
    cw = np.clip(np.sqrt(counts.sum() / (n_cls * counts)), 0.5, 3.0)
    class_weight = torch.tensor(cw, dtype=torch.float32, device=DEVICE)

    results = []
    for cfg in ABLATION_CONFIGS:
        result_path = os.path.join(_ROOT, cfg['result'])
        reuse = (RETRAIN_TAGS is not None and cfg['tag'] not in RETRAIN_TAGS
                 and os.path.isfile(result_path))
        if reuse:
            results.append(load_saved_ablation(cfg))
            continue
        results.append(run_single_ablation(
            cfg, train_ds, val_loader, test_loader, label_encoder,
            class_weight, train_loader
        ))

    print('\n' + '=' * 70)
    print('消融实验汇总')
    print('=' * 70)
    header = (f"{'组别':<22} {'GAT':<6} {'Mamba':<6} {'自蒸馏':<8} {'参数量(M)':>10} "
              f"{'最优轮':>6} {'Val-F1':>8} {'Test-F1':>8} {'Test-Acc':>8}")
    print(header)
    print('-' * 90)
    with open(SUMMARY_TXT, 'w', encoding='utf-8') as f:
        f.write('消融实验汇总（设备侧；结构对齐目录1）\n')
        f.write('A0=纯基线 | A1=先验动态门控GAT | A2=SimpleMamba+自蒸馏\n')
        f.write(f'window={WINDOW_SIZE} train_step={TRAIN_STEP} eval_step={EVAL_STEP} ')
        f.write(f'batch={BATCH_SIZE} lr={LR} patience={PATIENCE} dropout={DROPOUT}\n\n')
        f.write(header + '\n')
        f.write('-' * 90 + '\n')
        for r in results:
            line = (f"{r['name']:<22} {str(r['use_gat']):<6} {str(r['use_mamba']):<6} "
                    f"{str(r['use_mean_teacher']):<8} "
                    f"{r['params_M']:>10.3f} {r['best_epoch']:>6d} {r['val_f1']:>8.4f} "
                    f"{r['test_f1']:>8.4f} {r['test_acc']:>8.4f}")
            print(line)
            f.write(line + '\n')
    print(f'\n汇总结果已保存: {SUMMARY_TXT}')
    print('所有模型权重、分类报告、混淆矩阵均已保存到当前目录')


main()
